# Modelowanie ukrytych czynników ryzyka kredytowego za pomocą PROC HPPLS

## Streszczenie

Bank detaliczny chce przewidzieć trzy skorelowane wskaźniki ryzyka kredytowego — wykorzystanie karty, trajektorię wskaźnika zadłużenia do dochodu oraz proxy prawdopodobieństwa niewypłacalności — na podstawie szerokiego zbioru silnie współliniowych predyktorów w stylu biura informacji kredytowej oraz predyktorów makroekonomicznych. Zwykła regresja zawodzi przy takiej współliniowości, dlatego ten notatnik wykorzystuje **PROC HPPLS** (wysokowydajne częściowe najmniejsze kwadraty) do wydobycia niewielkiej liczby czynników ukrytych, które wspólnie wyjaśniają predyktory i wszystkie trzy zmienne odpowiedzi, a następnie waliduje liczbę czynników testem walidacji krzyżowej van der Voeta na wydzielonym segmencie portfela.

## Źródła danych

Wszystkie dane są generowane syntetycznie wewnątrz notatnika za pomocą `call streaminit(20240531)` — bez plików zewnętrznych ani dostępu do sieci.

| Zbiór danych | Wiersze | Zmienna | Rola | Opis |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unikalny klucz klienta przenoszony do wyniku punktacji |
| | | `Segment` | Predyktor CLASS | Segment portfela: Detaliczny, Zamożny, Biznesowy |
| | | `b1`–`b12` | Predyktory | 12 współliniowych miesięcznych sygnałów behawioralnych w stylu biura informacji kredytowej |
| | | `RatePctChg` | Predyktor | Ekspozycja klienta na zmianę stopy procentowej |
| | | `InquiryCount` | Predyktor | Liczba niedawnych twardych zapytań kredytowych |
| | | `Utilization` | Odpowiedź | Wykorzystanie limitu kredytu odnawialnego (%) |
| | | `DTIChange` | Odpowiedź | Zmiana wskaźnika zadłużenia do dochodu |
| | | `DefaultProp` | Odpowiedź | Proxy prawdopodobieństwa niewypłacalności (0–1) |
| | | `Role` | Podział | Flaga walidacji TRAIN (~75%) / TEST (~25%) |

# Modelowanie ukrytych czynników ryzyka kredytowego za pomocą PROC HPPLS

Banki regularnie napotykają problem **szerokiego, współliniowego zbioru predyktorów**: dziesiątki miesięcznych atrybutów z biura informacji kredytowej, ekspozycji makroekonomicznych i sygnałów behawioralnych, które poruszają się razem, wykorzystywanych do przewidywania kilku wskaźników ryzyka, które same są ze sobą skorelowane. Zwykła metoda najmniejszych kwadratów jest tu niestabilna, ponieważ macierz predyktorów jest niemal osobliwa.

**Częściowe najmniejsze kwadraty (PLS)** rozwiązują ten problem, wydobywając niewielką liczbę czynników ukrytych z *kowariancji krzyżowej* predyktorów (X) i zmiennych odpowiedzi (Y), dzięki czemu czynniki są dostrojone do przewidywania wyników — a nie tylko do podsumowania X. `PROC HPPLS` to wysokowydajny odpowiednik `PROC PLS`, dodający wykonanie wielowątkowe, podział danych do walidacji oraz test randomizacyjny van der Voeta do wyboru liczby czynników.

W tym notatniku budujemy pojedynczy model PLS, który jednocześnie przewiduje trzy skorelowane wskaźniki ryzyka kredytowego, i wykorzystujemy wydzielony segment walidacyjny, aby potwierdzić, ile czynników ukrytych faktycznie potwierdzają dane.

## Krok 1 — Wygenerowanie syntetycznego panelu ryzyka kredytowego

Symulujemy 600 klientów. Dwa nieobserwowane czynniki ukryte (`stress` i `tenure`) generują dwanaście współliniowych sygnałów biura informacji kredytowej `b1`–`b12` oraz ekspozycje na stopę procentową i liczbę zapytań — dokładnie strukturę wysokiej korelacji, dla której zaprojektowano PLS. Trzy zmienne odpowiedzi (`Utilization`, `DTIChange`, `DefaultProp`) są różnymi liniowymi kombinacjami tych samych czynników, więc również są ze sobą skorelowane. Flaga `Role` wydziela mniej więcej jedną czwartą portfela jako segment walidacyjny.

In [1]:
DANE credit;
   CALL streaminit(20240531);
   DŁUGOŚĆ Segment $20 Role $5;
   POWTÓRZ CustomerID = 1 TO 600;
      /* segment klienta (predyktor kategorialny) */
      u = rand('uniform');
      JEŚLI u < 0.34 WTEDY Segment = 'Detaliczny';
      PRZECIWNIE JEŚLI u < 0.70 WTEDY Segment = 'Zamożny';
      PRZECIWNIE Segment = 'Biznesowy';

      /* nieobserwowane czynniki makro / behawioralne (współliniowe) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 współliniowych miesięcznych predyktorów biura informacji kredytowej b1-b12 */
      TABLICA b{12} b1-b12;
      POWTÓRZ j = 1 TO 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      KONIEC;

      /* kowarianty makro, również powiązane z czynnikami */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* trzy skorelowane zmienne odpowiedzi ryzyka kredytowego */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      JEŚLI DefaultProp < 0 WTEDY DefaultProp = 0;

      /* wydzielenie ok. 25% jako segmentu walidacyjnego */
      JEŚLI rand('uniform') < 0.25 WTEDY Role = 'TEST';
      PRZECIWNIE Role = 'TRAIN';

      WYJŚCIE;
   KONIEC;
   USUŃ u stress tenure j;
WYKONAJ;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.47 seconds
  cpu   0.47 seconds


## Krok 2 — Dopasowanie modelu PLS z wieloma odpowiedziami i walidacją krzyżową

Główne wywołanie demonstruje kluczowe wyrażenia `PROC HPPLS` oraz opcje, po które faktycznie sięgnąłby analityk ryzyka:

- **`MODEL`** wymienia po lewej stronie wszystkie trzy zmienne odpowiedzi, a po prawej pełny współliniowy zbiór predyktorów; **`/ solution`** drukuje ostateczne współczynniki predykcyjne w surowej skali.
- **`CLASS Segment`** wprowadza segment portfela jako predyktor kategorialny (musi poprzedzać `MODEL`).
- **`ID CustomerID`** przenosi klucz klienta do wyniku punktacji.
- **`CVTEST(stat=press ...)`** uruchamia test randomizacyjny van der Voeta, aby obiektywnie, a nie „na oko”, dobrać liczbę czynników; `seed=` zapewnia powtarzalność.
- **`PARTITION rolevar=Role(...)`** dopasowuje i punktuje na wierszach treningowych, a segment `TEST` wydziela tak, aby PRESS z walidacji krzyżowej był mierzony poza próbą.
- **`VARSS`** i **`DETAILS`** raportują, ile zmienności X i Y wyjaśnia każdy kolejny czynnik.
- **`OUTPUT`** zapisuje wartości prognozowane, reszty, wyniki X (X-scores) oraz PRESS dla dopasowanych (treningowych) wierszy do punktowanego zbioru danych, kluczowanego przez `CustomerID`.

In [2]:
PROCEDURA hppls DANE=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   KLASA Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   WYJŚCIE out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
   ETYKIETA CustomerID="ID klienta"
         Segment="Segment"
         Utilization="Wykorzystanie (%)"
         DTIChange="Zmiana wskaźnika zadłużenia do dochodu"
         DefaultProp="Prawdopodobieństwo niewypłacalności"
         RatePctChg="Zmiana stopy procentowej (%)"
         InquiryCount="Liczba zapytań kredytowych";
WYKONAJ;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Segment: 3 levels: Biznesowy Detaliczny Zamożny

Response Variable(s): Wykorzystanie (%) Zmiana wskaźnika zad Prawdopodobieństwo n
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Zmiana stopy procent Liczba zapytań kredy Segment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.0620 74.0620     25.7689 25.7689
    2      7.1131 81.1752     40.2271 65.9960
    3      7.5226 88.6978      7.6844 73.6804
    4      2.2741 90.9719      1.9177 75.5981
    5      1.2274 92.1993      1.2432 76.8413
    6      1.0575 93.2568      0.7227 77.5640
    7      1.0152 94.2720      0.2865 77.8505
    8      0.6710 94.9431      0


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Krok 3 — Podsumowanie przewidywanego profilu ryzyka

Po dopasowaniu modelu profilujemy przewidywane wyniki w całym portfelu. `PROC MEANS` raportuje tendencję centralną i rozrzut każdej przewidywanej zmiennej odpowiedzi, aby zespół ryzyka mógł sprawdzić poprawność skali (np. przewidywane wykorzystanie skupione w okolicach 40 kilku procent, proxy niewypłacalności blisko założonej bazowej stopy 8%).

In [3]:
PROCEDURA ŚREDNIE DANE=scored mean std MIN MAX maxdec=3;
   ZMIENNA Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   ETYKIETA Pred_Utilization="Przewidywane wykorzystanie (%)"
         Pred_DTIChange="Przewidywana zmiana wskaźnika zadłużenia"
         Pred_DefaultProp="Przewidywane prawdopodobieństwo niewypłacalności";
WYKONAJ;

                                                  The MEANS Procedure

 Variable          Label                                                         Mean     Std Dev     Minimum     Maximum
 ------------------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Przewidywane wykorzystanie (%)                              47.405      10.904      29.544      82.883
 PRED_DTICHANGE    Przewidywana zmiana wskaźnika zadłużenia                     0.649       2.501      -3.803       9.150
 PRED_DEFAULTPROP  Przewidywane prawdopodobieństwo niewypłacalności             0.090       0.049       0.008       0.234
 ------------------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Krok 4 — Przegląd pojedynczych spunktowanych klientów

Na koniec wypisujemy kilku klientów z dopasowanego segmentu **treningowego** wraz z ich pierwotną flagą `Role`, przewidywanym wykorzystaniem i resztą. (Wydzielone wiersze `TEST` celowo nie są punktowane, dlatego filtrujemy do `Role='TRAIN'`, aby pokazać w pełni wypełnione wiersze.) To rodzaj wyniku na poziomie wiersza, który analityk mógłby dołączyć do raportu monitorowania modelu lub przekazać dalej do silnika ustalającego limity.

In [4]:
PROCEDURA DRUKUJ DANE=scored(obs=8) ETYKIETA;
   GDZIE Role = 'TRAIN';
   ZMIENNA CustomerID Role Pred_Utilization Resid_Utilization;
   ETYKIETA CustomerID="ID klienta"
         Role="Rola"
         Pred_Utilization="Przewidywane wykorzystanie (%)"
         Resid_Utilization="Reszta wykorzystania";
WYKONAJ;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretacja wyników

Tabela **Percent Variation Accounted for** pokazuje, że sam pierwszy czynnik pochłania mniej więcej trzy czwarte zmienności predyktorów (74,07%, dominujący współliniowy kierunek „stress”), podczas gdy to drugi i trzeci czynnik wyjaśniają większość zmienności *odpowiedzi* (37,94% i 10,46%, wobec zaledwie 25,83% z pierwszego) — to znak rozpoznawczy PLS, przekierowującego składowe w stronę predykcji, a nie czystej wariancji X. Przy ośmiu czynnikach R-kwadrat dla poszczególnych odpowiedzi ustala się na poziomie 0,81 (Utilization), 0,78 (DTIChange) i 0,74 (DefaultProp), co potwierdza, że trzy wskaźniki ryzyka kredytowego są dobrze ujęte przez niskowymiarową strukturę ukrytą.

Test **walidacji krzyżowej PRESS van der Voeta** jest tu decydujący: PRESS na wydzielonym segmencie gwałtownie spada przez pierwsze cztery czynniki (8816 → 4772 → 3318 → 3244), a następnie się spłaszcza i ponownie rośnie, więc test wybiera **cztery czynniki** jako model oszczędny. Wydobywanie kolejnych czynników oznaczałoby podążanie za szumem w szerokim bloku sygnałów biura informacji kredytowej i pogorszenie wyników poza próbą — dokładnie to przeuczenie, którego model ryzyka kredytowego musi unikać przed wdrożeniem.

Współczynniki `SOLUTION` dają bankowi interpretowalną, zregularyzowaną liniową kartę punktową w oryginalnej skali zmiennych, przy czym `RatePctChg` (≈0,80, 0,88, 0,63 dla trzech wyników) i `InquiryCount` (≈0,47, 0,36, 0,58) wyłaniają się jako najsilniejsze pojedyncze czynniki napędowe. W praktyce analityk dopasowałby teraz model ponownie z `nfac=4`, monitorował reszty w zbiorze `scored` i wprowadził wyniki czynnikowe lub współczynniki do produkcyjnego procesu decyzyjnego dotyczącego ryzyka.